
# Do LLMs know the difference between a pet chicken and a roast chicken?

## Word sense disambiguation in computational models and humans


In human language, words do not always have a fixed meaning. The most striking example is homonymous words: words that have the same form, but very different meanings. For instance, the word "bank", which has a different meaning in the context "I went to the bank to get some money" and "At the river bank, I met my old friend". Polysemous words are words that have different -- yet related -- meanings: for example, "chicken" is the same 'entity' in "My pet chicken is lovely" and "I am having roast chicken for dinner", but has very different meanings in these two contexts. In general, context can modulate almost any word's meaning. This poses a challenge in computational linguistics, as we need to find a way to differentiate among different meanings like humans do. Much research, resources, and models have been put forward to help with this challenge.

In this assignment, you are going to focus on [Trott and Bergen's (2021)](https://aclanthology.org/2021.acl-long.550/) RAW-C dataset: you are going to conduct a number of explorations with this dataset and partially replicate their research by the end of the assignment. In short, the authors explore how good LLMs are at capturing same/different meanings of words across contexts by comparing it to human judgements. To better understand the idea and the research, start by reading the paper.

This assignment entails a series of (interconnected) tasks (altogether worth 95 points):

* **Task 1**. Compute contextual word embeddings at different layers from Trott & Bergen's dataset. Here, each word is found in 4 sentences: 2 with one meaning, 2 with another meaning.
* **Task 2**. Compute sense embeddings for words in Trott & Bergen's dataset using WordNet, so you have an embedding for each definition of the word.
* **Task 3**. Compute the similarity between the contextual word embeddings of the homonyms at different layers and their sense embeddings; explore the relationship between homonyms and dominant senses quantitatively and qualitatively
* **Task 4**. Replicate part of Trott & Bergen's work by computing similarities across sentences with same/different meanings at the different layers and correlate with human similarities; visualise the results and reflect on them

In order to better understand the assignment, we recommend going through it all before starting so that it is clear how each part is connected to the next (which will help you make decisions about data structures, for instance).

# Task 1: Compute contextual word embeddings for homonyms [20 points]

## Task 1.1: read, explore and extract the necessary data [5 points]

First, you will have to (fork and) clone the github repository that stores the data you'll need. This can be found here: https://github.com/sashakenjeeva/raw-c . The repo also includes a README with a description of the original files in the repository, as well as some notes relevant for this assignment specifically.

Make sure you mount the drive now so that you have access to the folder (think about setting the working directory in a way that is convenient).

In [365]:
import os
from pathlib import Path

# Create a mount to the project root
PROJECT_ROOT = Path(os.getcwd())

print(f"=== Project root: ===\n{PROJECT_ROOT}\n")

# Create a data path
DATA_PATH = PROJECT_ROOT / "data"

if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_PATH, exist_ok=True)

print(f"=== Data path: ===\n{DATA_PATH}\n")

=== Project root: ===
/Users/kazikgarstecki/Desktop/university/comp_ling_assignment
=== Data path: ===
/Users/kazikgarstecki/Desktop/university/comp_ling_assignment/data


Now, you will have to read the data and organise it in a structure that works for the next parts of the assignment.

Read and explore the dataframe to see its structure (print part of it). What we need from it are the homonyms (in the form that they appear in the sentence -- the lexeme -- and in their regular form -- the lemma) and their corresponding sentences with different meanings (M1_a and M1_b have same meaning; M2_a, M2_b have same meaning). We only will need the stimuli that are in the final RAW-C dataset, as this is what we'll replicate at the end.

You can decide which data structure to use, but make sure that all these pieces of information are there (the word, the string, the meaning id, and the corresponding sentences) and easy to retrieve. Show your data at the end, as well as how many stimuli you end up with.

In [367]:
import pandas as pd

# Load the dataset
df_rawc = pd.read_csv(DATA_PATH / "raw-c.csv")

# Explore the dataset
print(f"=== Dataframe shape: ===\n{df_rawc.shape}\n")
print(f"=== Dataframe types: ===\n{df_rawc.dtypes}\n")

=== Dataframe shape: ===
(672, 20)

=== Dataframe types: ===
word                        str
sentence1                   str
sentence2                   str
same                       bool
ambiguity_type              str
disambiguating_word1        str
disambiguating_word2        str
version                     str
Class                       str
mean_relatedness        float64
median_relatedness      float64
diff                    float64
count                     int64
sd_relatedness          float64
distance_bert           float64
distance_elmo           float64
se_relatedness          float64
v1                          str
v2                          str
string                      str
dtype: object



In [375]:
print(f"=== Dataframe description: ===")
df_rawc.info()

=== Dataframe description: ===
<class 'pandas.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   word                  672 non-null    str    
 1   sentence1             672 non-null    str    
 2   sentence2             672 non-null    str    
 3   same                  672 non-null    bool   
 4   ambiguity_type        672 non-null    str    
 5   disambiguating_word1  672 non-null    str    
 6   disambiguating_word2  672 non-null    str    
 7   version               672 non-null    str    
 8   Class                 672 non-null    str    
 9   mean_relatedness      672 non-null    float64
 10  median_relatedness    672 non-null    float64
 11  diff                  672 non-null    float64
 12  count                 672 non-null    int64  
 13  sd_relatedness        672 non-null    float64
 14  distance_bert         672 non-null    float64
 15  dis

In [377]:
# Declare important columns
columns_of_interest = [
    "word", 
    "string",
    "sentence1",
    "sentence2",
    "v1",
    "v2",
]

# Filter for said columns and rename them
df_rawc_filtered = df_rawc[columns_of_interest] 
df_rawc_filtered = df_rawc_filtered.rename(columns={"word": "lemma", "string": "word"})

df_rawc_filtered.head(5)

,lemma,word,sentence1,sentence2,v1,v2
0,act,act,It was a desperate act.,It was a magic act.,M1_a,M2_a
1,act,act,It was a desperate act.,It was a comedic act.,M1_a,M2_b
2,act,act,It was a humane act.,It was a magic act.,M1_b,M2_a
3,act,act,It was a humane act.,It was a comedic act.,M1_b,M2_b
4,act,act,It was a desperate act.,It was a humane act.,M1_a,M1_b


In [456]:
df_rawc_sentence_split = (
    pd.concat(
        [   
            # Take the 1st version/sentences 
            df_rawc_filtered[["lemma", "word", "sentence1", "v1"]].rename(
                columns={"sentence1": "sentence", "v1": "v"}
            ),
            # Take the 2nd version/sentences 
            df_rawc_filtered[["lemma", "word", "sentence2", "v2"]].rename(
                columns={"sentence2": "sentence", "v2": "v"}
            )
        ]
    )
    # Drop duplicates
    .drop_duplicates(subset=["lemma", "word", "sentence"])
    .reset_index(drop=True)
)

# Sort values for the sake of visualization
df_rawc_sentence_split.sort_values(axis=0, by=["lemma", "v"], inplace=True, ignore_index=True)

print(f"=== Dataframe shape: ===\n{df_rawc_sentence_split.shape}\n")
print(f"=== Unique items: ===\n{df_rawc_sentence_split.nunique()}\n")

PROCESSED_DATA = DATA_PATH / "processed"
if not os.path.exists(PROCESSED_DATA):
    os.makedirs(PROCESSED_DATA, exist_ok=True)
    
df_rawc_sentence_split.to_csv(PROCESSED_DATA / "raw_c_sentence_split.csv", index=False)

# Inspect the dataframe
df_rawc_sentence_split.head(5)

=== Dataframe shape: ===
(448, 4)

=== Unique items: ===
lemma       112
word        112
sentence    448
v             4
dtype: int64



,lemma,word,sentence,v
0,act,act,It was a desperate act.,M1_a
1,act,act,It was a humane act.,M1_b
2,act,act,It was a magic act.,M2_a
3,act,act,It was a comedic act.,M2_b
4,appeal,appeal,He had a universal appeal.,M1_a


## Task 1.2: Compute the contextualised word embeddings [15 points]


Now that you have the homonyms and their corresponding sentences, we will need to compute word embeddings for each of them. For this we will use the BERT base model, in its uncased version.

That is, for each homonym, you will have to compute four embeddings: one for the homonym in M1_a, one in M1_b, one in M2_a, one in M2_b. However, we also want to look into different layers of the BERT model to see which one captures the homonym's meaning best: you want to calculate embeddings at the static layer and at layers 4, 8, 12.

We will use the package psycho-embeddings (you will use it in class), which allows us to specify which target words we want to obtain the embeddings of, in which sentences, and at which layers, among other things. Make sure to read the documentation of the package so that you know the meaning of the arguments and which ones will come useful to you.

First of all, install the psycho-embeddings package below.

In [393]:
# install the psycho-embeddings package here

# Clone the repository
if not os.path.exists(PROJECT_ROOT / "psycho-embeddings"):
    !git clone https://github.com/MilaNLProc/psycho-embeddings

# Uncomment depending on the package manager you are using
# !pip install -e . 
!uv pip install -e .

Resolved 34 packages in 13ms                                         
   Building comp-ling-assignment @ file:///Users/kazikgarstecki/Desktop/university/c
   Building comp-ling-assignment @ file:///Users/kazikgarstecki/Desktop/university/c
   Building comp-ling-assignment @ file:///Users/kazikgarstecki/Desktop/university/c
   Building comp-ling-assignment @ file:///Users/kazikgarstecki/Desktop/university/c
   Building comp-ling-assignment @ file:///Users/kazikgarstecki/Desktop/university/c
   Building comp-ling-assignment @ file:///Users/kazikgarstecki/Desktop/university/c
      Built comp-ling-assignment @ file:///Users/kazikgarstecki/Desktop/university/c
Prepared 1 package in 1.19s                                              
Uninstalled 1 package in 1ms
Installed 1 package in 2mst==0.1.0 (from file:///Users/kazik
 ~ comp-ling-assignment==0.1.0 (from file:///Users/kazikgarstecki/Desktop/university/comp_ling_assignment)


Now, import the relevant module/function from psycho-embeddings and load the required BERT model.

In [594]:
# your code here
from psycho_embeddings import ContextualizedEmbedder

# Load model
bert_model = ContextualizedEmbedder("bert-base-uncased", max_length=128)

loading configuration file config.json from cache at /Users/kazikgarstecki/.cache/huggingface/hub/models--bert-base-uncased/snapshots/86b5e0934494bd15c9632b12f734a8a67f723594/config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_hidden_states": true,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.5.4",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.

Now, test that everything works correctly by computing an embedding for the word "assignment" in the sentence "I am having so much fun with this assignment!", at static layer and layers 4, 8 and 12 (hint: think of tokenisation and how the embedder deals with that).

In [595]:
import numpy as np

test_embeddings = bert_model.embed(
    words=["assignment"],
    target_texts=["I am having so much fun with this assignment!"],
    layers_id=[4, 8, 12],
    batch_size=8,
    return_static=True,
)

layer_idx = [(-1, "static"), (4, "4"), (8, "8"), (12, "12")]

for idx, layer in layer_idx:
    print(f"\n=== Layer {layer} Embedding: ===")
    emb = test_embeddings[idx][0]
    print(f"type: {type(emb)}")
    print(f"shape: {emb.shape}")

  0%|          | 0/1 [00:00<?, ?it/s]/Users/kazikgarstecki/Desktop/university/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 1/1 [00:00<00:00, 15.51it/s]


=== Layer static Embedding: ===
type: <class 'numpy.ndarray'>
shape: (768,)

=== Layer 4 Embedding: ===
type: <class 'numpy.ndarray'>
shape: (768,)

=== Layer 8 Embedding: ===
type: <class 'numpy.ndarray'>
shape: (768,)

=== Layer 12 Embedding: ===
type: <class 'numpy.ndarray'>
shape: (768,)


The next step is to calculate embeddings for the homonyms and their sentences that we got from the RAW-C dataset.

Make sure that your final output includes the word, the meaning id (M1_a, etc), the corresponding sentence and the embeddings at static layer and layers 4, 8, 12. You should maximally optimise this process by calculating in batches (again, check psycho-embeddings documentation), but keep in mind this might still take a while. First test your pipeline with a small number of inputs, and only run the full scale embedding extraction once you're positive the code works as expected.

When done, save the output in [pickle](https://docs.python.org/3/library/pickle.html) format (this is similar to json, but it can also handle np.arrays), so that you can easily load it later when needed and do not have to run it again. After pickle dumping (that's the word for saving it in pickle format), print it so that you are sure everything was saved correctly.

Then, check that your final data includes everything that you need by checking the entry "bank" and print the data pertaining to "bank".

In [596]:
# Calculate embeddings for homonyms and their sentences
test_raw_c_embeddings = bert_model.embed(
    words = df_rawc_sentence_split["word"].tolist(),
    target_texts = df_rawc_sentence_split["sentence"].tolist(),
    layers_id = [4, 8, 12],
    batch_size = 32,
    return_static = True
)

# Map emb dict keys to column names
emb_key_layer_mapping = {
    4: "layer_4",
    8: "layer_8",
    12: "layer_12",
    -1: "layer_static"
}

# Ensure no changes are applied to previous dataframe
df_rawc_with_embeddings = df_rawc_sentence_split.copy(deep=True)

# Include embeddings as part of the new dataframe
for key, layer in emb_key_layer_mapping.items():
    df_rawc_with_embeddings[layer] = test_raw_c_embeddings[key]

# Save as pickle
df_rawc_with_embeddings.to_pickle(PROCESSED_DATA / "raw_c_with_embeddings.pkl")
# Load to ensure everything went well
df_rawc_with_embeddings = pd.read_pickle(PROCESSED_DATA / "raw_c_with_embeddings.pkl")

# Confirm that embeddings are saved as numpy arrays
random_embedding = df_rawc_with_embeddings.iloc[0]["layer_static"]
assert isinstance(random_embedding, np.ndarray)

# Inspect the new dataframe
df_rawc_with_embeddings.head(5)

  0%|          | 0/14 [00:00<?, ?it/s]/Users/kazikgarstecki/Desktop/university/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 14/14 [00:07<00:00,  1.75it/s]


,lemma,word,sentence,v,layer_4,layer_8,layer_12,layer_static
0,act,act,It was a desperate act.,M1_a,"[1.6829053, 0.08441312, -0.33193615, -0.341306...","[1.0654582, -0.36254317, 0.1188567, 0.13084409...","[0.3789006, -0.21466061, -0.057178605, -0.1951...","[0.017291509, 0.01293602, -0.053804133, -0.061..."
1,act,act,It was a humane act.,M1_b,"[1.5745126, -0.53692955, -0.36310813, -0.64165...","[1.5665528, -0.40207732, 0.26410484, 0.0026419...","[0.30742064, -0.411638, -0.10526247, -0.319348...","[0.017291509, 0.01293602, -0.053804133, -0.061..."
2,act,act,It was a magic act.,M2_a,"[1.9441345, 0.028225563, -0.33929366, -0.99731...","[1.6207538, -0.3879069, 0.40122306, -0.4561103...","[0.9379027, 0.039944846, 0.2086345, -0.2862577...","[0.017291509, 0.01293602, -0.053804133, -0.061..."
3,act,act,It was a comedic act.,M2_b,"[0.86945885, -0.2438263, 0.01976711, -0.857342...","[0.78628355, -0.5734637, 0.53356946, -0.123716...","[0.06930962, -0.06950791, -0.2661462, -0.33356...","[0.017291509, 0.01293602, -0.053804133, -0.061..."
4,appeal,appeal,He had a universal appeal.,M1_a,"[1.0649076, -0.9980851, -0.43838885, -1.073057...","[0.73011875, 0.18825388, 0.60285467, -0.466076...","[0.46518275, -0.09599658, 0.2225516, -0.109499...","[-0.028775984, -0.080165826, -0.023797046, -0...."


# Task 2: Compute sense embeddings for the homonym dataset using WordNet [20 points]

Your next task is to fetch the definitions (glosses) of the homonyms, and compute an embedding for each gloss (each gloss is associated with a specific sense). We do that so we can later see whether the contextualised embeddings computed above represent the meaning of the homonym in context well (by comparing it to the sense embeddings). Figure 18.9 in [Jurafsky's and Martin's (2021) chapter 18](https://web.stanford.edu/~jurafsky/slp3/old_sep21/18.pdf) graphically illustrates this idea. Use this chapter for this part of the assignment, as it will come useful for you both theoretically and practically.

## Task 2.1: Fetch senses and glosses for a word [5 points]

First of all, you will have to figure out how [WordNet](https://www.nltk.org/howto/wordnet.html) works within the nltk package (hint: pay attention to what a synset is).

Install and import all the necessary components and define a function to extract the glosses of a word and create a dictionary with senses and glosses.

Then use the word "bat" to test that everything is working correctly: i.e., for "bat", you should be able to get its senses and the gloss for each of the sense (you will see that synsets might contain related words, but you only need the senses that contain the word of interest or derivates thereof; this should be specified in the function). Print the output for "bat".


In [641]:
# your code here
import nltk

# Download wordnet
nltk.download("wordnet")
from nltk.corpus import wordnet as wn

def get_senses_and_glosses(word):
    temp_dict = {
        "word": [],
        "synset": [],
        "gloss": [],
    }

    for synset in wn.synsets(word):
        # Retreive the gloss
        gloss = synset.definition()

        sense = synset.name()

        # Check if the sense 
        if sense.split(".")[0].startswith(word):
            temp_dict["word"].append(word)
            temp_dict["synset"].append(sense)
            temp_dict["gloss"].append(gloss)
    
    return temp_dict

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/kazikgarstecki/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [642]:
# Retrieve bat synsets
bat_synsets = get_senses_and_glosses("bat")

# Print bat output
for synset, gloss in zip(bat_synsets["synset"], bat_synsets["gloss"]):
    print(f"{synset}: {gloss}")

bat.n.01: nocturnal mouselike mammal with forelimbs modified to form membranous wings and anatomical adaptations for echolocation by which they navigate
bat.n.02: (baseball) a turn trying to get a hit
bat.n.05: a club used for hitting a ball in various games
bat.v.01: strike with, or as if with a baseball bat
bat.v.02: wink briefly
bat.v.03: have a turn at bat
bat.v.04: use a bat


## Task 2.2: Function to compute sense embeddings [10 points]

Now that you have a function to extract senses and glosses for a given word, write a function that takes a word and computes embeddings for each of the senses following the method explained in Jurafsky's and Martin's chapter. In this case, no need to calculate at different layers: you should use the last layer only. You should maximally optimise this function like before.

The output should include the sense, the gloss, and the embedding. Print the function's output when using the word "bank".


In [690]:
def retrieve_sense_embeddings(
    lemma_list: str | list[str], 
    get_senses_and_glosses_func = get_senses_and_glosses, 
    model: ContextualizedEmbedder = bert_model,
    LAST_LAYER_IDX: int = 12
):
    all_lemmas = []
    all_glosses = []
    all_synsets = []
    words_to_embed = []
    target_texts_to_embed = []    

    if isinstance(lemma_list, str):
        lemma_list = [lemma_list]
    
    for lemma in lemma_list:
        # Retrieve the senses and glosses
        lemma_synset = get_senses_and_glosses_func(lemma)
        synsets, glosses = lemma_synset["synset"], lemma_synset["gloss"]
        all_lemmas.extend([lemma] * len(glosses))
        all_glosses.extend(glosses)
        all_synsets.extend(synsets)

        # Append them to the lists later passed to the embedder
        words_to_embed.extend([synset.split(".")[0] for synset in synsets])
        # NOTE: the target texts will always follow the form: f"{sense}: {gloss}"
        target_texts_to_embed.extend([f"{sense.split(".")[0]}: {gloss}" for sense, gloss in zip(synsets, glosses)])

    sense_embeddings = model.embed(
        words=words_to_embed,
        target_texts=target_texts_to_embed,
        layers_id=[LAST_LAYER_IDX],
        batch_size=16,
        return_static=False
    )
    
    return {
        "lemma": all_lemmas,
        "synset": all_synsets,
        "sense": words_to_embed,
        "gloss": all_glosses,
        "embedding": sense_embeddings[LAST_LAYER_IDX]
    }

In [691]:
# Run the sense embeddding function on the word "bank"
bank_sense_embeddings = retrieve_sense_embeddings("bank")

# Visualize the output
pd.DataFrame().from_dict(bank_sense_embeddings)

  0%|          | 0/1 [00:00<?, ?it/s]/Users/kazikgarstecki/Desktop/university/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 1/1 [00:00<00:00,  3.96it/s]


,lemma,synset,sense,gloss,embedding
0,bank,bank.n.01,bank,sloping land (especially the slope beside a bo...,"[0.12150506, -0.14082775, -0.24302498, -0.2460..."
1,bank,bank.n.03,bank,a long ridge or pile,"[0.031304292, -0.22398716, -0.5094195, -0.4397..."
2,bank,bank.n.04,bank,an arrangement of similar objects in a row or ...,"[0.31977126, -0.23580891, -0.7017689, 0.142542..."
3,bank,bank.n.05,bank,a supply or stock held in reserve for future u...,"[0.42835185, -0.33511797, -0.57390106, 0.07775..."
4,bank,bank.n.06,bank,the funds held by a gambling house or the deal...,"[-0.0022778758, 0.012502015, -0.72262394, -0.0..."
5,bank,bank.n.07,bank,a slope in the turn of a road or track; the ou...,"[-0.041520525, -0.21230876, -0.29962787, -0.32..."
6,bank,bank.n.09,bank,a building in which the business of banking tr...,"[0.4822492, 0.019960396, -0.3508716, -0.172362..."
7,bank,bank.n.10,bank,a flight maneuver; aircraft tips laterally abo...,"[-0.04142214, -0.5214778, -0.1963114, -0.21985..."
8,bank,bank.v.01,bank,tip laterally,"[-0.17739385, -0.20971015, -0.47318688, -0.137..."
9,bank,bank.v.02,bank,enclose with a bank,"[0.28713334, 0.033689097, -0.33909184, -0.1865..."


## Task 2.3: Compute sense embeddings for the RAW-C stimuli [5 points]

Now, use the function you defined above to compute sense embeddings for the RAW-C stimuli and pickle dump it too.

As above, the information that should be there for each word is: the sense, the gloss, the embedding at the last layer. Again, you can think of which structure to use best, but keep in mind that we will have to compare these to the CWE calculated in task 1, so it is good to think of a similar structure that is easily comparable.

Make sure that the number of stimuli matches the number of stimuli in the final RAW-C dataset.

In [694]:
# Retrieve unique lemmas
raw_c_lemmas = df_rawc_with_embeddings["lemma"].unique().tolist()
# Make sure the number of stimuli checks out 
assert len(raw_c_lemmas) == 112

# Extract sense embeddings for RAW-C stimuli
raw_c_sense_embeddings = retrieve_sense_embeddings(raw_c_lemmas)
# Create a dataframe from the function output
df_raw_c_sense_embeddings = pd.DataFrame().from_dict(raw_c_sense_embeddings)

# Make sure the number of stimuli after checks out
raw_c_sense_lemmas = df_raw_c_sense_embeddings["lemma"].unique().tolist()
assert len(raw_c_lemmas) == len(raw_c_sense_lemmas)

# Save the sense embeddings as a pickle
df_raw_c_sense_embeddings.to_pickle(PROCESSED_DATA / "raw_c_sense_embeddings.pkl")

  0%|          | 0/64 [00:00<?, ?it/s]/Users/kazikgarstecki/Desktop/university/comp_ling_assignment/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 64/64 [00:17<00:00,  3.71it/s]


# Task 3: Compute and explore similarity between homonym CWEs and sense embeddings [35 points]

You now have the homonym CWEs computed in task 1, and the sense embeddings computed in task 2. The next step is to calculate cosine similarities between each CWE for each homonym (at the selected layer!) and each sense embedding for that homonym.

For instance, say for the word "bat" with meaning M1_a, you have its CWE at the static layer and at layers 4, 8, 12 and 7 senses: here, you will end up with 16 cosine similarities (take each CWE and compute its similarity to each of the sense embeddings). We then want to see which sense meaning is the closest to each CWE, and do some qualitative explorations with that.

## Task 3.1: Compute the cosine similarity between all the CWEs and the sense embeddings [8 points]

This task is not trivial with regards to how much information you have and how to structure the data (this is why it's also important to think of data structures in the earlier parts of the assignment), so take some time to think how to best breakdown this task. Test each step/function if you have multiple. Pickle dump your final output so that it is easily retrievable for later. At the end, print an example of the entry "bank".

For cosine similarity, the cdist function from scipy.spatial.distance seems the most efficient, but you are free to use any of your liking (hint: pay attention to the shape of your embeddings and to similarity vs distance. You will need the similarity).

In [708]:
from sklearn.metrics.pairwise import cosine_similarity

lemma_list = []
word_list = []
sentence_list = []
v_list = []
cwe_layer_list = []
cwe_emb_list = []
synset_list = []
sense_list = []
gloss_list = []
sense_emb_list = []
cos_sim_list = []

for _, row in df_rawc_with_embeddings.iterrows():
    # Load data into vars
    lemma = row["lemma"]
    word = row["word"]
    sentence = row["sentence"]
    v = row["v"]

    cwe_embeddings = {
        "static": row["layer_static"],
        "4": row["layer_4"],
        "8": row["layer_8"],
        "12": row["layer_12"]
    }

    # Retrieve sense embeddings for relevant lemma
    sense_filter = df_raw_c_sense_embeddings["lemma"] == row["lemma"]
    sense_emb_df = df_raw_c_sense_embeddings[sense_filter]

    synsets, senses, glosses, sense_embeddings = sense_emb_df["synset"], sense_emb_df["sense"].tolist(), sense_emb_df["gloss"].tolist(), sense_emb_df["embedding"].tolist()

    # Compare each CWE for each lemma meaning to all sense embeddings and store the cosine similarities
    for ly, cwe_emb in cwe_embeddings.items():
        for synset, sense, gloss, sense_emb in zip(synsets, senses, glosses, sense_embeddings):
            reshaped_cwe_emb, reshaped_sense_emb = cwe_emb.reshape(1, -1), sense_emb.reshape(1, -1)
            cos_sim = cosine_similarity(reshaped_cwe_emb, reshaped_sense_emb).item()

            lemma_list.append(lemma)
            word_list.append(word)
            sentence_list.append(sentence)
            v_list.append(v)
            cwe_layer_list.append(ly)
            cwe_emb_list.append(cwe_emb)
            synset_list.append(synset)
            sense_list.append(sense)
            gloss_list.append(gloss)
            sense_emb_list.append(sense_emb)
            cos_sim_list.append(cos_sim)

# Create the dataframe from the results
df_raw_c_cwe_sense_emb_cos_similarities = pd.DataFrame({
    "lemma": lemma_list,
    "word": word_list,
    "sentence": sentence_list,
    "v": v_list,
    "cwe_layer": cwe_layer_list,
    "cwe_emb": cwe_emb_list,
    "synset": synset_list,
    "sense": sense_list,
    "gloss": gloss_list,
    "sense_emb": sense_emb_list,
    "cos_sim": cos_sim_list
})

# Save the dataframe as a pickle
df_raw_c_cwe_sense_emb_cos_similarities.to_pickle(PROCESSED_DATA / "raw_c_cwe_sense_emb_cos_similarities.pkl")

# Inspect the data
df_raw_c_cwe_sense_emb_cos_similarities.head()

,lemma,word,sentence,v,cwe_layer,cwe_emb,synset,sense,gloss,sense_emb,cos_sim
0,act,act,It was a desperate act.,M1_a,static,"[0.017291509, 0.01293602, -0.053804133, -0.061...",act.n.01,act,a legal document codifying the result of delib...,"[0.78000736, -0.5004632, -0.35106277, -0.74719...",0.142451
1,act,act,It was a desperate act.,M1_a,static,"[0.017291509, 0.01293602, -0.053804133, -0.061...",act.n.02,act,something that people do or cause to happen,"[0.43565655, -0.38396898, -0.6966581, -0.20040...",0.151373
2,act,act,It was a desperate act.,M1_a,static,"[0.017291509, 0.01293602, -0.053804133, -0.061...",act.n.03,act,a subdivision of a play or opera or ballet,"[-0.26154134, -0.401941, -0.029744823, -0.6416...",0.224635
3,act,act,It was a desperate act.,M1_a,static,"[0.017291509, 0.01293602, -0.053804133, -0.061...",act.n.04,act,a short theatrical performance that is part of...,"[0.20125636, -0.4539132, -0.36038178, -0.40815...",0.173200
4,act,act,It was a desperate act.,M1_a,static,"[0.017291509, 0.01293602, -0.053804133, -0.061...",act.n.05,act,a manifestation of insincerity,"[0.3635703, -0.16950282, -0.36749712, -0.19376...",0.135901


## Task 3.2: Quantitative and qualitative explorations the relationship between homonym embeddings and dominant senses

Now, we can look into how the CWEs in different meanings and layers relate to the different senses of a homonym. We'll focus on the dominant sense in WordNet, see below for more details. This section includes both code blocks and reflection questions.

### Dominant senses in WordNet and top senses across layers (focus on static layer) [8 points]

Embeddings at the static layer do not take into account context, so intuitively they should capture the 'average' meaning, maybe the most common/dominant. We can test this by looking at the most similar sense and seeing if that matches that most common/dominant sense in the synset.

Keep in mind that synsets mark more common/dominant senses with numbering: so n.01 will be the most common noun; v.01 the most common verb, etc. If that is not available, the most common meaning will be the next number (e.g., n.02). You have to take that into account when you extract the top sense, so first extract information about which are the most dominant senses for each word across all the parts of speech: for example, "bat" might have as its two most common senses bat.n.01 and bat.v.02 (because v.01 might not be available; this is just an example). Some words might only have one part of speech in their synset, some more. Print your results.

In [327]:
# your code here
parts_of_speech = ['n', 'v', 's', 'a', 'r']

def extract_dominant_senses(word: str, parts_of_speech: list[str] = parts_of_speech):
    dominant_pos = {}
    for synset in wn.synsets(word):
        name_split = synset.name().split(".")
        pos, n = name_split[1], int(name_split[2])
        if pos not in dominant_pos.keys():
            dominant_pos[pos] = (n, synset.definition())
        elif n < dominant_pos[pos][0]:
            dominant_pos[pos] = (n, synset.definition())

    formatted_dict = {
        pos: definition for pos, (_, definition) in zip(dominant_pos.keys(), dominant_pos.values())
    }  
    
    for pos in parts_of_speech:
        if pos not in formatted_dict.keys():
            formatted_dict[pos] = None

    return formatted_dict

for pos, dom_sense in extract_dominant_senses("bat").items():
    print(f"{pos}: {dom_sense}")

n: nocturnal mouselike mammal with forelimbs modified to form membranous wings and anatomical adaptations for echolocation by which they navigate
v: strike with, or as if with a baseball bat
s: None
a: None
r: None


In [328]:
df_sentences_copy[raw_c_sense_embeddings["lemma"] == "bat"].shape[0]

/var/folders/66/0hdjd2q91lg9ptg7znk80kk40000gn/T/ipykernel_33439/1252171209.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_sentences_copy[raw_c_sense_embeddings["lemma"] == "bat"].shape[0]


10

In [329]:
lemmas = df_sentences_copy["lemma"].unique().tolist()
n_dominant = []
v_dominant = []
s_dominant = []
a_dominant = []
r_dominant = []

for lemma in lemmas:
    pos_dominant_senses = extract_dominant_senses(lemma)
    for pos, pos_list in zip(parts_of_speech, [
        n_dominant, v_dominant, s_dominant, a_dominant, r_dominant
    ]):
        pos_list.append(pos_dominant_senses[pos])

dominant_senses_df = pd.DataFrame(
    {
        "lemma": lemmas,
        "n_dominant": n_dominant,
        "v_dominant": v_dominant,
        "s_dominant": s_dominant,
        "a_dominant": a_dominant,
        "r_dominant": r_dominant,
    }
)

dominant_senses_df

,lemma,n_dominant,v_dominant,s_dominant,a_dominant,r_dominant
0,act,a legal document codifying the result of delib...,"perform an action, or work out or perform (an ...",NaN,NaN,NaN
1,appeal,earnest or urgent request,take a court case to a higher court for review,NaN,NaN,NaN
2,atmosphere,a particular environment or surrounding influence,NaN,NaN,NaN,NaN
3,bail,(criminal law) money that must be forfeited by...,release after a security has been paid,NaN,NaN,NaN
4,band,a group of musicians playing popular music for...,"bind or tie together, as with a band",NaN,NaN,NaN
...,...,...,...,...,...,...
107,title,a heading that names a statute or legislative ...,designate by an identifying term,NaN,NaN,NaN
108,toast,slices of bread that have been toasted,make brown and crisp by heating,NaN,NaN,NaN
109,toll,a fee levied for the use of roads or bridges (...,ring slowly,NaN,NaN,NaN
110,track,a course over which races are run,carry on the feet and deposit,NaN,NaN,NaN


Then, extract the top similarity of homonyms to the senses at all the layers you have available. While we are interested in the static layer for checking dominant senses, it is also interesting to look into other layers to see whether adding context will refine the captured meaning.


In [330]:
dominant_senses_df[dominant_senses_df["lemma"] == "act"]

,lemma,n_dominant,v_dominant,s_dominant,a_dominant,r_dominant
0,act,a legal document codifying the result of delib...,"perform an action, or work out or perform (an ...",NaN,NaN,NaN


In [331]:
# your code here
df_sentences_sense_copy = df_sentences_copy.copy(deep=True)

def is_most_common_sense(lemma: str, gloss: str, dominant_senses_df: pd.DataFrame = dominant_senses_df):
    domninant_senses_row = dominant_senses_df[dominant_senses_df["lemma"] == lemma]
    dominant_pos_senses = [domninant_senses_row[pos_col].item() for pos_col in dominant_senses_df.columns if pos_col != "lemma"]
    if gloss in dominant_pos_senses:
        return True
    return False

LAYERS = ["static", "layer_4", "layer_8", "layer_12"]

for layer_id in LAYERS:
    df_sentences_sense_copy[f"{layer_id}_is_dom_sense"] = df_sentences_sense_copy.apply(
        lambda row: is_most_common_sense(row["lemma"], row[f"{layer_id}_gloss_match"]), axis=1
    )

df_sentences_sense_copy[[f"{ly}_is_dom_sense" for ly in LAYERS]].sum() / df_sentences_sense_copy[[f"{ly}_is_dom_sense" for ly in LAYERS]].shape[0]

static_is_dom_sense      0.241071
layer_4_is_dom_sense     0.180804
layer_8_is_dom_sense     0.185268
layer_12_is_dom_sense    0.247768
dtype: float64

Let's check an example from our results.

Out of all the similarities of 'bank' to all its senses at all the layers, which one is the highest? Print your results for that entry and reflect below.

In [332]:
# your code here

### Does the static layer capture the most dominant meaning, according to WordNet (and according to you)? [2 point]

%your answer here

### Across other layers and meanings, which layer seems to capture the meaning of bank across meanings best, and why do you make this conclusion? [2 points]

%your answer here

### Checking matches and mismatches with the dominant sense [5 points]

Now, let's quantitatively check if the static layer actually captures the most dominant sense (any POS). You should end up with two data structures: matches (when the most similar sense is one of the dominant senses) and mismatches (when the most similar sense is not one of the dominant sense). Do that also for the other layers to compare. Print the percentage of matches and mismatches per layer.



In [333]:
# your code here

Now, print the matches and mismatches for the static layer only.

In [334]:
# your code here

### Do BERT's static embeddings capture the most dominant sense in WordNet? [2 point]

%your answer here

### Do the percentages of matches and mismatches throughout the layers make sense to you or is it different than what you expected? [2 points]

%your answer here

### For the **static layer**, are there any words that seem to particularly deviate from the dominant meaning? If so, which and why could that be? [3 points]

%your answer here

### Do you think the corpus on which BERT is trained might reflect different meaning dominance than for WordNet's senses? If so/not, why? [3 points]

%your answer here

# Task 4: Partially replicate Trott & Bergen's experiment [20 points]

Now comes the time to partially replicate the RAW-C experiment, by seeing whether different layers of BERT capture meanings more or less similarly to humans. At the end you will have to wrap up with a brief comment on which layer seems to capture meanings best and how that connects to explorations in the previous section.

## Task 4.1: Create a dataframe with cosine similarities between sentences at different layers [7 points]

You should now use the embeddings at the different layers that you computed to calculate similarities between each context: M1a, M1b, M2a, M2b. You will have to have all combinations, so for each string in the RAW-C dataframe, you'll have: M1a vs M1b, M1a vs M2a, M1a vs M2b, M1b vs M2a, M1b vs M2b, M2a vs M2b.

Bear in mind that your final dataframe should include: the word, the string as it appears in the sentence, cosine similarity at layers 4, layer 8, layer 12, the version being compared (is it M1a vs M1b or M1a vs M2a?) and the mean relatadness given by humans (hint: the repo you cloned will come useful here, both in terms of code and data). Print the head of the dataframe to check everything is in order, and check also that the number of stimuli match with your number across the assignment (starting from task 1).

In [335]:
# your code here

## Task 4.2: Correlate with human judgements and visualise [8 points]

First, correlate the cosine similarities at the different layers to the mean human relatedness judgements. Use the same correlation metric used by Trott & Bergen.

In [336]:
# your code here

Next, visualise your results. You want to see the correlation between BERT embeddings and human judgements per layer, but what would also be interesting is to include the meaning contrasts (such as M1_a_M1_b, etc), so that we can see how those play out per layer.

In [337]:
# your code here

### Reflect on the correlations and on the visualisations. What can you observe and infer in terms of which layer(s) might be capturing meaning best? Is there one way to determine that (i.e., what does 'capturing meanings' mean?)? Contrast and compare the layers. [5 points]

%your answer here



